# LAND SURFACE PREDICTION

---

**Summary of LST Prediction Pipeline:**
1.  **Part 1**: Ingested 3 years of Landsat data and built the **UHI Composite** and **Seasonality** features.
2.  **Part 2**: Established that linear models (Ridge) couldn't handle the non-linear urban heat dynamics.
3.  **Part 3**: Optimized a **Conservative XGBoost** model that achieved a validation $R^2$ of **0.5101** with a Mean Absolute Error of **$2.38^{\circ}C$**.

---

### **Part 1: Foundation & Physics-Informed Feature Engineering**This part covers the environment setup, data ingestion, and the crucial transformation of raw satellite bands into the 5 balanced features that drive your models

 **Block 1: Library Imports & Global Setup**We start by importing the necessary data science and machine learning libraries used in your study, including `XGBoost` for your final optimized models

* **Environment Setup**: Loads the necessary tools for data manipulation (Pandas/NumPy), visualization (Matplotlib/Seaborn), and machine learning (Scikit-learn/XGBoost).
* **Path Configuration**: Defines the specific file locations for the 2021, 2022, and 2023 Landsat datasets

In [10]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn utilities
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold, cross_val_score

# Gradient Boosting
from xgboost import XGBRegressor

print("🚀 LST PREDICTION PROJECT: PART 1 - DATA & FEATURE ENGINEERING")
print("=" * 60)

# Configuration: Update these paths to match your local or Kaggle environment
TRAIN_FILES = [
    'samples_clean_consistent_LULC_2021.csv',
    'samples_clean_consistent_LULC_2022.csv'
]
VALIDATION_FILE = 'samples_clean_consistent_LULC_2023.csv'

🚀 LST PREDICTION PROJECT: PART 1 - DATA & FEATURE ENGINEERING


---

### **Block 2: Data Loading Utility**This block handles the ingestion of your Landsat 8 and 9 datasets. It includes a sampling parameter to manage computational load while preserving spatial diversity

* **Multi-Temporal Ingestion**: Combines satellite data from different years to build a robust training foundation.
* **Managed Sampling**: Uses a random sampling strategy to ensure the dataset is representative but computationally manageable

In [11]:
def load_data(file_paths, target_samples=None):
    """Loads and combines CSV datasets from different years."""
    dfs = []
    for file in file_paths:
        try:
            df = pd.read_csv(file)
            if target_samples and len(df) > target_samples:
                # Random sampling to maintain manageable data volume
                df = df.sample(target_samples, random_state=42)
            dfs.append(df)
        except FileNotFoundError:
            print(f"⚠️ Warning: {file} not found. Check your paths.")

    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)

# [cite_start]Loading 2021-2022 for training and 2023 for future validation [cite: 642, 644]
train_raw = load_data(TRAIN_FILES, target_samples=150000)
val_raw = load_data([VALIDATION_FILE], target_samples=150000)

print(f"📊 Training records (2021-22): {len(train_raw):,}")
print(f"📊 Validation records (2023):  {len(val_raw):,}")

📊 Training records (2021-22): 300,000
📊 Validation records (2023):  150,000


---

### **Block 3: Physics-Informed Feature Engineering**This is the core "Phase 2" of your research. Here, we calculate the cyclical seasonal driver, the vegetation cooling effect, and the Urban Heat Composite (UHI) to provide robust physical signals to the models

* **Cyclical Seasonality ($Season_{cos}$)**: Transforms the month into a cosine wave to ensure the model understands that December and January are temporally close.
* **Vegetation Cooling ($Effect_{veg}$)**: Combines NDVI and SAVI into a single term to eliminate redundant data (multicollinearity) while keeping the cooling signal.
* **Urban Heat Composite ($Composite_{UHI}$)**: Creates a custom interaction term between built-up intensity, low reflectivity (Albedo), and low vegetation to identify peak heat zones

In [12]:
def create_balanced_features(df):
    """
    Transforms raw indices into 5 physics-informed features
    to address multicollinearity and seasonal cycles.
    """
    df = df.copy()

    # [cite_start]1. Cyclical Seasonal Encoding (Season_cos) [cite: 664]
    # Corrects the mathematical gap between December and January
    df['season_cos'] = np.cos(2 * np.pi * (df['month'] - 1) / 12)

    # [cite_start]2. Vegetation Cooling Effect (Effect_veg) [cite: 669]
    # Combines NDVI and SAVI to solve high correlation (r=0.98)
    df['vegetation_effect'] = df['NDVI'] * df['SAVI']

    # [cite_start]3. Urban Heat Composite (Composite_UHI) [cite: 674]
    # Models interaction between Built-up, Low Albedo, and Low Vegetation
    df['urban_heat_composite'] = df['NDBI'] * (1 - df['Albedo']) * (1 - df['NDVI'])

    return df

print("🎯 Engineering balanced feature set...")
train_df = create_balanced_features(train_raw)
val_df = create_balanced_features(val_raw)

# [cite_start]The 5 features defined in the final modeling methodology [cite: 677]
balanced_features = [
    'NDBI',
    'vegetation_effect',
    'Albedo',
    'season_cos',
    'urban_heat_composite'
]

🎯 Engineering balanced feature set...


---

### **Block 4: Data Splitting & Scaling**
Finally, we separate the target variable (LST) from the predictors and apply standardization. Note that we fit the scaler **only** on training data to prevent data leakage

* **Strict Temporal Splitting**: Separates data by year (2021–22 for training, 2023 for validation) to simulate a real-world "future" forecast.
* **Standardization**: Scales features so they have a mean of 0 and a standard deviation of 1, fitting only on training data to prevent "data leakage"

In [13]:
# Feature/Target Separation
X_train = train_df[balanced_features]
y_train = train_df['LST']

X_val = val_df[balanced_features]
y_val = val_df['LST']

# [cite_start]Preserve coordinates for spatial visualization later [cite: 597]
train_coords = train_raw[['lon', 'lat', 'LST']].copy()
val_coords = val_raw[['lon', 'lat', 'LST']].copy()

# [cite_start]Scaling (Crucial for Ridge Regression) [cite: 686]
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("✅ Part 1 Completed: Data is scaled and features are engineered.")
print(f"📋 Modeling with features: {balanced_features}")

✅ Part 1 Completed: Data is scaled and features are engineered.
📋 Modeling with features: ['NDBI', 'vegetation_effect', 'Albedo', 'season_cos', 'urban_heat_composite']


---

Continuing with the structured implementation of your project, **Part 2** focuses on the first phase of your modeling methodology: comparing the four distinct algorithm families to establish a performance baseline.

---

## **Part 2: Multi-Algorithm Comparison & Performance Evaluation**
This part executes the training for the linear, bagging, and boosting models using the 5-feature set engineered in Part 1

### **Block 5: Ridge Regression (Linear Baseline)**
* **Linear Foundation**: Implements Ridge Regression to provide a stable, regularized linear baseline for LST prediction.
* **Complexity Control**: Uses a complexity parameter $\alpha$ to prevent any single feature from disproportionately influencing the results.
* **Optimization**: Iterates through multiple $\alpha$ candidates to find the version with the highest validation $R^{2}

In [14]:
print("\n1. 🏔️ RIDGE REGRESSION (LINEAR BASELINE)")
print("-" * 40)

# Grid search for optimal alpha (regularization strength)
alpha_candidates = [0.1, 1.0, 5.0, 10.0, 20.0]
best_alpha = 5.0
best_val_r2 = -float('inf')

for alpha in alpha_candidates:
    ridge_test = Ridge(alpha=alpha, random_state=42)
    ridge_test.fit(X_train_scaled, y_train)
    r2_test = r2_score(y_val, ridge_test.predict(X_val_scaled))
    if r2_test > best_val_r2:
        best_val_r2 = r2_test
        best_alpha = alpha

# Train final Ridge model with best alpha
ridge_final = Ridge(alpha=best_alpha, random_state=42)
ridge_final.fit(X_train_scaled, y_train)

# Store results
results = {}
results['Ridge'] = {
    'train_r2': r2_score(y_train, ridge_final.predict(X_train_scaled)),
    'val_r2': best_val_r2,
    'val_rmse': np.sqrt(mean_squared_error(y_val, ridge_final.predict(X_val_scaled))),
    'val_mae': mean_absolute_error(y_val, ridge_final.predict(X_val_scaled)),
    'model': ridge_final, 'scaler': scaler,
    'val_pred': ridge_final.predict(X_val_scaled)
}
print(f"✅ Optimal Alpha: {best_alpha} | Validation R²: {best_val_r2:.4f}")


1. 🏔️ RIDGE REGRESSION (LINEAR BASELINE)
----------------------------------------
✅ Optimal Alpha: 0.1 | Validation R²: 0.3495


---

### **Block 6: Random Forest (Bagging Ensemble)**
* **Variance Reduction**: Employs a multitude of decision trees on bootstrapped data samples to reduce prediction variance.
* **Non-Parametric Approach**: Capable of capturing non-linear interactions without requiring feature scaling.
* **Noise Robustness**: Uses averaging to maintain stability against the typical noise found in satellite-derived spectral indices

In [15]:
print("\n2. 🌲 RANDOM FOREST (BAGGING ENSEMBLE)")
print("-" * 40)

rf = RandomForestRegressor(
    n_estimators=150,
    max_depth=20,
    min_samples_split=15,
    min_samples_leaf=8,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
rf_val_pred = rf.predict(X_val)

results['RandomForest'] = {
    'train_r2': r2_score(y_train, rf.predict(X_train)),
    'val_r2': r2_score(y_val, rf_val_pred),
    'val_rmse': np.sqrt(mean_squared_error(y_val, rf_val_pred)),
    'val_mae': mean_absolute_error(y_val, rf_val_pred),
    'model': rf, 'scaler': None,
    'val_pred': rf_val_pred
}
print(f"✅ Training R²: {results['RandomForest']['train_r2']:.4f} | Validation R²: {results['RandomForest']['val_r2']:.4f}")


2. 🌲 RANDOM FOREST (BAGGING ENSEMBLE)
----------------------------------------
✅ Training R²: 0.6495 | Validation R²: 0.4880


---

### **Block 7: XGBoost & AdaBoost (Boosting Models)**
* **XGBoost (Sequential Learning)**: Builds trees sequentially, with each new tree correcting the residual errors of the previous ones.
* **AdaBoost (Adaptive Capacity)**: Focuses training capacity on "hard" examples that were previously mispredicted.
* **Error Correction**: Uses specialized additive strategies to minimize a regularized objective function

In [17]:
print("\n3. ⚡ XGBOOST & 📈 ADABOOST (BOOSTING MODELS)")
print("-" * 40)

# XGBoost: Phase 1 version
xgb_p1 = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.08, random_state=42, n_jobs=-1, tree_method='hist')
xgb_p1.fit(X_train, y_train)
xgb_val_pred = xgb_p1.predict(X_val)

results['XGBoost'] = {
    'train_r2': r2_score(y_train, xgb_p1.predict(X_train)),
    'val_r2': r2_score(y_val, xgb_val_pred),
    'val_rmse': np.sqrt(mean_squared_error(y_val, xgb_val_pred)),
    'val_mae': mean_absolute_error(y_val, xgb_val_pred),
    'model': xgb_p1, 'scaler': None,
    'val_pred': xgb_val_pred
}

# AdaBoost: Focus on difficult samples
ada = AdaBoostRegressor(n_estimators=150, learning_rate=0.1, random_state=42)
ada.fit(X_train, y_train)
ada_val_pred = ada.predict(X_val)

results['AdaBoost'] = {
    'train_r2': r2_score(y_train, ada.predict(X_train)),
    'val_r2': r2_score(y_val, ada_val_pred),
    'val_rmse': np.sqrt(mean_squared_error(y_val, ada_val_pred)),
    'val_mae': mean_absolute_error(y_val, ada_val_pred),
    'model': ada, 'scaler': None,
    'val_pred': ada_val_pred
}
print(f"✅ XGBoost Val R²: {results['XGBoost']['val_r2']:.4f} | AdaBoost Val R²: {results['AdaBoost']['val_r2']:.4f}")


3. ⚡ XGBOOST & 📈 ADABOOST (BOOSTING MODELS)
----------------------------------------
✅ XGBoost Val R²: 0.5026 | AdaBoost Val R²: 0.3430


---

### **Block 8: Metrics Calculation & Comparison Table**
* **$R^2$ Score**: Quantifies the proportion of LST variance that the model successfully explains.
* **RMSE & MAE**: Measures the standard deviation of errors and the linear average error magnitude in degrees Celsius.
* **Generalization Gap**: Calculates the difference between training and validation $R^2$ to detect overfitting

In [18]:
print("\n" + "="*75)
print("📊 COMPREHENSIVE PERFORMANCE COMPARISON (PHASE 1)")
print("="*75)
print(f"{'Algorithm':15} {'Train R²':10} {'Val R²':10} {'Gap':8} {'Val RMSE':12}")
print("-" * 65)

for algo, metrics in results.items():
    gap = metrics['train_r2'] - metrics['val_r2']
    print(f"{algo:15} {metrics['train_r2']:10.4f} {metrics['val_r2']:10.4f} {gap:8.4f} {metrics['val_rmse']:10.3f}°C")

# Coordinate mapping for visualization
for name, info in results.items():
    val_coords[f'LST_pred_{name}'] = info['val_pred']

print("\n✅ Part 2 Completed: Initial models are evaluated and stored.")


📊 COMPREHENSIVE PERFORMANCE COMPARISON (PHASE 1)
Algorithm       Train R²   Val R²     Gap      Val RMSE    
-----------------------------------------------------------------
Ridge               0.2859     0.3495  -0.0636      3.585°C
RandomForest        0.6495     0.4880   0.1616      3.181°C
XGBoost             0.5544     0.5026   0.0518      3.135°C
AdaBoost            0.4068     0.3430   0.0638      3.603°C

✅ Part 2 Completed: Initial models are evaluated and stored.


## **Part 3: Advanced Optimization & Final Visualizations**
This part handles the fine-tuning of the boosting architecture, model persistence, and the generation of report-ready spatial heatmaps.

### **Block 9: Advanced XGBoost Optimization (Phase 2)**
* **Variant Testing**: Implements the three specific configurations from your study: Baseline, DART (Dropout), and Conservative.
* **Stability Focus**: The **XGB_Conservative** variant uses shallower trees ($depth=5$) and more estimators ($300$) to force generalization on noisy satellite data.
* **Cross-Validation**: Uses 5-fold cross-validation to ensure the performance is consistent across different data subsets.

In [20]:
print("\n🏆 PHASE 2: ADVANCED XGBOOST OPTIMIZATION (CHAPTER 7)")
print("=" * 60)

# Define the specific variants tested in your report
xgb_variants = {
    # Variant 2: Dropout Additive Regression Trees (DART)
    'XGB_DART': XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.08,
        subsample=0.8, colsample_bytree=0.8, booster='dart',
        rate_drop=0.1, skip_drop=0.5, random_state=42, n_jobs=-1, tree_method='hist'
    ),
    # Variant 3: Conservative - THE FINAL OPTIMAL MODEL
    'XGB_Conservative': XGBRegressor(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.7, colsample_bytree=0.7, reg_alpha=2.0,
        reg_lambda=2.0, min_child_weight=5, random_state=42, n_jobs=-1, tree_method='hist'
    )
}

xgb_results = {}
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in xgb_variants.items():
    # Perform cross-validation as mentioned in your methodology
    cv_scores = cross_val_score(model, X_train, y_train, cv=kfold, scoring='r2')

    # Fit and predict for 2023 Validation
    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)

    xgb_results[name] = {
        'train_r2': r2_score(y_train, model.predict(X_train)),
        'val_r2': r2_score(y_val, val_pred),
        'val_rmse': np.sqrt(mean_squared_error(y_val, val_pred)),
        'val_mae': mean_absolute_error(y_val, val_pred),
        'model': model, 'val_pred': val_pred
    }
    print(f"✅ {name:16} | CV R²: {np.mean(cv_scores):.4f} | Val R²: {xgb_results[name]['val_r2']:.4f}")

# Update coordinate dataset with advanced predictions
for name, result in xgb_results.items():
    val_coords[f'LST_pred_{name}'] = result['val_pred']


🏆 PHASE 2: ADVANCED XGBOOST OPTIMIZATION (CHAPTER 7)


KeyboardInterrupt: 

---

### **Block 10: Model Selection & Persistence**
* **Best Model Selection**: Automatically identifies the model with the highest $R^2$ on the unseen 2023 data.
* **Persistence**: Saves the final model, the feature scaler, and the performance metadata into a single `.joblib` file for future deployment.
* **Data Export**: Exports the validation dataset with latitude/longitude and model predictions for GIS analysis.

In [ ]:
print("\n💾 FINALIZING MODEL & SAVING RESULTS")
print("-" * 40)

# Merge Phase 1 and Phase 2 results to find the absolute winner
all_models = {**results, **xgb_results}
best_algo = max(all_models.items(), key=lambda x: x[1]['val_r2'])
best_name = best_algo[0]
best_info = best_algo[1]

print(f"🏆 Overall Best Model: {best_name}")
print(f"📈 Final Validation R²: {best_info['val_r2']:.4f}")

# Save the final package
final_package = {
    'model': best_info['model'],
    'scaler': best_info['scaler'],
    'features': balanced_features,
    'performance': best_info
}

joblib.dump(final_package, 'final_lst_prediction_model.joblib')
val_coords.to_csv('final_predictions_with_coordinates.csv', index=False)
print("✅ Saved: final_lst_prediction_model.joblib & final_predictions_with_coordinates.csv")

---

### **Block 11: Final Report Visualizations**
* **Spatial Heatmap**: Generates a side-by-side comparison of Actual LST vs. Predicted LST across the Mangaluru geographic extent.
* **Generalization Bar Chart**: Plots Training vs. Validation $R^2$ to visually demonstrate the "Generalization Gap" for all models.
* **Residual Analysis**: Plots Actual vs. Predicted LST to verify the $R^2$ and RMSE performance.

In [ ]:
print("\n🗺️ GENERATING FINAL SPATIAL VISUALIZATIONS")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: Actual Ground Truth
sc1 = ax1.scatter(val_coords['lon'], val_coords['lat'], c=val_coords['LST'], cmap='hot', s=2, alpha=0.6)
ax1.set_title('Actual LST (Mangaluru 2023)', fontweight='bold')
plt.colorbar(sc1, ax=ax1, label='Celsius')

# Subplot 2: Best Model Prediction (XGB_Conservative)
pred_col = f'LST_pred_{best_name}'
sc2 = ax2.scatter(val_coords['lon'], val_coords['lat'], c=val_coords[pred_col], cmap='hot', s=2, alpha=0.6)
ax2.set_title(f'Predicted LST - {best_name}', fontweight='bold')
plt.colorbar(sc2, ax=ax2, label='Celsius')

plt.tight_layout()
plt.show()

# Final Performance Gap Chart (Mirroring Report Figure 7.3)
plot_data = []
for name, m in all_models.items():
    plot_data.append({'Model': name, 'R2': m['train_r2'], 'Type': 'Training'})
    plot_data.append({'Model': name, 'R2': m['val_r2'], 'Type': 'Validation'})

plt.figure(figsize=(12, 6))
sns.barplot(data=pd.DataFrame(plot_data), x='Model', y='R2', hue='Type')
plt.title('Model Generalization: Training vs. Validation R²', fontweight='bold')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.show()

print("\n🚀 PROJECT COMPLETE: All code blocks are intact and results are saved!")

---